# 06.12 — EMA prior normalization deep dive

[06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb) introduced the mechanism: divide each loss by its running EMA magnitude, rescale to classification's, weight, sum. This companion goes *under the hood* into two things 06.4 deferred: the exact **order of operations** (the normalize step divides by the *updated* prior, which produces a striking first-iteration degeneracy), and the **cross-component dynamics** — what happens over many batches as the components improve at different rates and the single shared reference ties them together.

This notebook covers:

* The exact order: EMA-update the prior *first*, then divide by it.
* First-iteration degeneracy — every component's *value* collapses to `rescale × weight`, but the *gradient* is still normalized.
* The transient: a fast-improving component normalizes *below* the reference (self-rebalancing).
* The shared-reference cross-component coupling and the fallback chain.

**Prerequisites:** [06.4 EMA prior normalization](06.4_the_ema_prior_normalization_deep_dive.ipynb).

## Section 1 — What MATLAB does

`cgg_processLossComponent` (lines 216-274) updates the prior and normalizes in a specific order:

```matlab
new_prior    = raw .* (1 - PriorProportion) + old_prior .* PriorProportion;  % EMA FIRST
normalized   = raw ./ new_prior;         % divide by the UPDATED prior
rescaled     = normalized .* Rescale;    % Rescale = classification's prior
weighted     = rescaled .* Weight;
```

The order matters: the prior is EMA-updated *before* it's used as the divisor. On the very first iteration there's no history, so `new_prior = raw` — and `raw ./ new_prior = 1` exactly, for *every* component. `cgg_getLossInformation` picks `Rescale` (classification's prior, or reconstruction's as fallback). The port reproduces this precisely; the empirical probe ([06.10](06.10_nan_masked_reconstruction.ipynb)) is the parity authority.

## Section 2 — The Python concepts you need

### 2.1 — The order: update the prior, *then* divide by it

The subtlety 06.4 skipped: `_process_component` doesn't divide by the *old* prior — it EMA-updates the prior first, then divides the loss by the *new* one. So the divisor always includes a `(1 − proportion)` slice of the *current* batch. At steady state this barely matters; on batch 1 it's everything, because with no history the new prior *is* the current loss.

### 2.2 — First-iteration degeneracy: the value collapses

In [ ]:
import torch
from neural_data_decoding.training.losses.multi_objective import aggregate_normalized_losses, LossPriors

w = {"reconstruction": 2.0, "kl": 3.0, "classification": 5.0}
print("Same weights, WILDLY different raw reconstruction — watch the normalized output:")
for recon_mag in (10.0, 1000.0, 0.01):
    out = aggregate_normalized_losses(
        reconstruction_loss=torch.tensor(recon_mag), kl_loss=torch.tensor(7.0),
        classification_loss=torch.tensor(4.0), weights=w, priors=LossPriors.initial())
    print(f"  recon raw={recon_mag:>8}: recon_out={out.reconstruction.item():.2f}, "
          f"kl_out={out.kl.item():.2f}, class_out={out.classification.item():.2f}")
print("→ recon_out is 8.00 EVERY time (= rescale 4.0 × weight 2.0). The raw magnitude fully cancels.")

On the first batch, every component's *value* is exactly `rescale × weight` — the raw loss magnitude cancels completely, because `raw / new_prior = raw / raw = 1`. The total is `rescale × Σ(weights)`, independent of how big or small any individual loss actually is. This is the **first-iteration degeneracy**. It's not a bug — it's the normalization doing its job maximally: on batch 1, with no history to smooth against, it perfectly equalizes every component to the reference scale.

### 2.3 — …but the gradient is *not* degenerate

In [ ]:
# The VALUE degenerates, but the gradient carries the 1/magnitude normalization:
for mag in (10.0, 1000.0):
    r = torch.tensor(mag, requires_grad=True)
    out = aggregate_normalized_losses(
        reconstruction_loss=r, classification_loss=torch.tensor(4.0),
        weights={"reconstruction": 2.0, "classification": 5.0}, priors=LossPriors.initial())
    out.reconstruction.backward()
    print(f"recon raw={mag:>7}: value={out.reconstruction.item():.2f} (same), "
          f"gradient={r.grad.item():.5f} (= rescale·weight/magnitude = 8/{mag})")
print("→ big-magnitude component → proportionally SMALLER gradient. That's the point: gradient balance.")

This is the crux of why normalization works. The *value* being `rescale × weight` is cosmetic; what trains the model is the *gradient*, and the gradient is `∂loss/∂θ × rescale·weight / magnitude`. A component with a huge raw loss (magnitude 1000) gets its gradient divided by 1000 — so it can't bulldoze the others just for being numerically large ([06.1 §2.2](06.1_multi_task_losses_overview.ipynb)). Each component's gradient is scaled to a common footing, and the *weights* set the balance. The division by the loss's own magnitude is precisely a per-component gradient normalization.

### 2.4 — The transient: fast-improving components down-weight themselves

In [ ]:
import matplotlib.pyplot as plt

priors = LossPriors.initial()
raws, prior_vals, outs = [], [], []
for batch in range(8):
    recon_mag = 100.0 * (0.6 ** batch)          # reconstruction improving each batch
    out = aggregate_normalized_losses(
        reconstruction_loss=torch.tensor(recon_mag), classification_loss=torch.tensor(4.0),
        weights={"reconstruction": 2.0, "classification": 5.0}, priors=priors, prior_proportion=0.9)
    priors = out.updated_priors
    raws.append(recon_mag); prior_vals.append(priors.reconstruction.item()); outs.append(out.reconstruction.item())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(raws, "o-", label="raw reconstruction loss")
ax1.plot(prior_vals, "s--", label="EMA prior (lags)")
ax1.set_title("raw drops fast; the EMA prior lags behind"); ax1.set_xlabel("batch"); ax1.legend()
ax2.plot(outs, "d-", color="crimson", label="normalized recon_out")
ax2.axhline(8.0, ls=":", color="gray", label="rescale×weight = 8 (steady state)")
ax2.set_title("normalized output dips BELOW the reference"); ax2.set_xlabel("batch"); ax2.legend()
plt.tight_layout(); plt.show()
print("recon_out:", [round(v, 2) for v in outs])
print("→ improving faster than its own EMA average → normalizes below rescale×weight → gets down-weighted.")

06.4 said normalization brings each component to "≈1.0." That's the **steady state** — true when the raw loss sits at its EMA average. The deep-dive truth: during a *transient*, the EMA prior lags the raw loss, so a component that's improving faster than its own recent average normalizes to *less* than `rescale × weight`. Its contribution to the total shrinks, so the optimizer's effective attention shifts toward components that *haven't* improved yet. It's an emergent self-rebalancing: whichever component is making the fastest progress temporarily yields the floor to the laggards. (A *worsening* component does the reverse — it normalizes above the reference and gets more attention.)

### 2.5 — The shared reference couples all components

Every component is rescaled by the *same* `Rescale_Value` — classification's prior ([06.4 §2.3](06.4_the_ema_prior_normalization_deep_dive.ipynb)). That single shared factor is what makes the components *commensurable*: normalize each to its own ~1.0, then stamp them all with classification's scale so "1 unit of reconstruction" and "1 unit of KL" mean the same thing. The fallback chain (`_determine_rescale_value`): classification if active → reconstruction if not (a pure autoencoder, [05.6](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb) Stage 1) → `1.0` if neither. Change what's active and the whole reference frame shifts — which is correct, because in Stage 1 there's no classification to anchor to.

## Section 3 — The `neural_data_decoding` implementation

`_process_component` — the exact order: EMA-update, then divide by the new prior, rescale, weight:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/losses/multi_objective.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "raw_detached = loss.detach()" in line)
for k in range(i, i + 26):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* Step 1 EMA-updates the prior; on the first iteration (`current_prior is None`) it falls to `new_prior = raw_detached` — the bootstrap that causes the §2.2 degeneracy.
* Step 2 divides by `new_prior` (the *updated* one, §2.1), guarded by `(new_prior != 0).all()` — a zero prior (a genuinely-zero component) skips normalization rather than dividing by zero.
* Step 3 multiplies by the shared `rescale_value`; step 4 by the per-component `weight`; step 5 by the optional confidence `beta` ([06.7](06.7_the_confidence_pd_controller.ipynb)).
* `raw_detached` feeds the *prior* (detached — a constant), but `loss_working` starts from the live `loss`, so the **gradient flows through the numerator** while the prior/rescale/weight are constants ([02.5 §2.4](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb)). That's the §2.3 gradient-survives-degeneracy in code.
* `_determine_rescale_value` (just above) implements the §2.5 fallback chain and returns a **detached** reference.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the degeneracy is exact

Confirm that on the first iteration, `component_out / (rescale × weight) == 1.0` for every component, whatever the raw losses.

In [ ]:
# Reveal:
import random
out = aggregate_normalized_losses(
    reconstruction_loss=torch.tensor(random.uniform(1, 1000)),
    kl_loss=torch.tensor(random.uniform(0.01, 5)),
    classification_loss=torch.tensor(3.3),
    weights={"reconstruction": 2.0, "kl": 4.0, "classification": 1.0},
    priors=LossPriors.initial())
rescale = out.rescale_value.item()
print("recon_out / (rescale·2) =", round(out.reconstruction.item() / (rescale * 2.0), 6))
print("kl_out    / (rescale·4) =", round(out.kl.item() / (rescale * 4.0), 6))
print("→ exactly 1.0: batch-1 every component normalizes to the reference, raw magnitude irrelevant.")

### Exercise 4.2 — the reference is classification's own magnitude

Show that classification's *own* normalized output on batch 1 equals `classification_loss × weight` — the reference maps (nearly) back to its own scale.

In [ ]:
# Reveal:
class_loss, class_w = 3.3, 6.0
out = aggregate_normalized_losses(
    reconstruction_loss=torch.tensor(50.0), classification_loss=torch.tensor(class_loss),
    weights={"reconstruction": 1.0, "classification": class_w}, priors=LossPriors.initial())
print(f"class_out = {out.classification.item():.3f}   vs   class_loss × weight = {class_loss * class_w:.3f}")
print("→ equal: since rescale = class_loss and class normalizes to 1.0, class_out = class_loss × weight.")

## Section 5 — Common errors

### Expecting each component to stay pinned at `rescale × weight`

Only at steady state (§2.4). During transients the EMA prior lags, so improving components dip below and worsening ones rise above. If you see components drifting off the reference, that's the self-rebalancing working, not a bug.

### Dividing by the *old* prior

The order is EMA-update-then-divide (§2.1). Dividing by the pre-update prior changes the transient dynamics and breaks first-iteration parity (you'd get `raw/old_prior`, not `1.0`). Match MATLAB's order.

### A zeroed component blows up or vanishes

The `(new_prior != 0).all()` guard (§3) skips normalization for a genuinely-zero prior. If a component's normalized value looks un-scaled, its prior may be zero — check whether that component is actually active.

### The reference frame silently changed

Enabling/disabling classification flips `Rescale_Value` between classification's and reconstruction's magnitude (§2.5). Weights tuned under one reference behave differently under the other — the Stage 1 → Stage 2 handoff ([05.6](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb)) crosses exactly this boundary.

### Priors not threaded → permanent first-iteration degeneracy

If every call gets a fresh `LossPriors.initial()` ([06.4 §5](06.4_the_ema_prior_normalization_deep_dive.ipynb)), *every* batch is "batch 1" — every component permanently pinned to `rescale × weight`, the EMA never engages. Thread `updated_priors` forward.

## Section 6 — Further reading

- [06.4 EMA prior normalization](06.4_the_ema_prior_normalization_deep_dive.ipynb) — the introduction this notebook deepens.
- [`src/neural_data_decoding/training/losses/multi_objective.py`](../../src/neural_data_decoding/training/losses/multi_objective.py) — `_process_component`, `_determine_rescale_value`.
- [06.1 multi-task losses overview](06.1_multi_task_losses_overview.ipynb) — the scale problem this whole mechanism solves.

Next up: **[06.13 sampling layer deterministic at inference](06.13_sampling_layer_deterministic_at_inference.ipynb)** — the last Module 06 notebook: why the VAE is stochastic in training but deterministic at eval.